# Stage 1 Idiom Signal Training on Colab

This notebook trains the Stage-1 idiom/literal classifier **M** on the `unique_mwe_target_label_language.csv` file.

Output:

```text
idiom_signal_40ep.zip
```

Download this zip and copy/extract it into the project as:

```text
stage1_idiom_signal/output/
```

In [ ]:
!pip -q install transformers sentencepiece scikit-learn tqdm

In [ ]:
from google.colab import files

uploaded = files.upload()
CSV_PATH = next(iter(uploaded.keys()))
print('uploaded:', CSV_PATH)

In [ ]:
from __future__ import annotations

import csv
import json
import random
import re
import shutil
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = 'xlm-roberta-base'
OUTPUT_DIR = Path('/content/idiom_signal_40ep')
ZIP_PATH = Path('/content/idiom_signal_40ep.zip')
EPOCHS = 40
BATCH_SIZE = 16
ENCODE_BATCH_SIZE = 32
LEARNING_RATE = 1e-4
MAX_LENGTH = 128
VAL_RATIO = 0.15
SEED = 42
DROPOUT = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def mark_mwe_in_sentence(sentence: str, mwe: str) -> str:
    if not mwe:
        return sentence
    match = re.search(re.escape(mwe), sentence, flags=re.IGNORECASE)
    if match is None:
        return sentence
    start, end = match.span()
    return f'{sentence[:start]}<mwe>{sentence[start:end]}</mwe>{sentence[end:]}'


def build_idiom_input(sentence: str, mwe: str, language: str) -> str:
    marked_sentence = mark_mwe_in_sentence(sentence=sentence, mwe=mwe)
    return f'[LANG={language}] {marked_sentence} [MWE] {mwe}'


def read_rows(csv_path: str | Path) -> list[dict[str, Any]]:
    rows = []
    with Path(csv_path).open('r', encoding='utf-8-sig', newline='') as file:
        reader = csv.DictReader(file)
        required = {'Language', 'MWE', 'Target', 'Label'}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'missing columns: {sorted(missing)}')
        for row in reader:
            language = (row.get('Language') or '').strip()
            mwe = (row.get('MWE') or '').strip()
            target = (row.get('Target') or '').strip()
            label = float((row.get('Label') or '').strip())
            rows.append({
                'language': language,
                'mwe': mwe,
                'target': target,
                'label': label,
                'text': build_idiom_input(target, mwe, language),
            })
    if not rows:
        raise ValueError('CSV has no rows.')
    return rows


def split_train_val(labels: list[float], val_ratio: float, seed: int) -> tuple[list[int], list[int]]:
    rng = random.Random(seed)
    by_label: dict[float, list[int]] = {}
    for index, label in enumerate(labels):
        by_label.setdefault(label, []).append(index)

    train_indices = []
    val_indices = []
    for group in by_label.values():
        shuffled = group[:]
        rng.shuffle(shuffled)
        val_count = max(1, round(len(shuffled) * val_ratio))
        val_count = min(val_count, len(shuffled) - 1)
        val_indices.extend(shuffled[:val_count])
        train_indices.extend(shuffled[val_count:])

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    return train_indices, val_indices


def mean_pool(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

In [ ]:
class IdiomSignalModel(nn.Module):
    def __init__(self, model_name: str = MODEL_NAME, dropout: float = DROPOUT) -> None:
        super().__init__()
        self.model_name = model_name
        self.dropout = dropout
        self.encoder_frozen = True
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_dim = self.encoder.config.hidden_size
        classifier_hidden_dim = max(1, hidden_dim // 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, 1),
        )
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False
        self.encoder.eval()

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            hidden_states = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            ).last_hidden_state
        pooled = mean_pool(hidden_states, attention_mask)
        return self.classifier(pooled)

In [ ]:
set_seed(SEED)
rows = read_rows(CSV_PATH)
labels = [row['label'] for row in rows]
train_indices, val_indices = split_train_val(labels, VAL_RATIO, SEED)

print('rows:', len(rows))
print('train:', len(train_indices))
print('val:', len(val_indices))
print('label counts:', {label: labels.count(label) for label in sorted(set(labels))})

In [ ]:
model = IdiomSignalModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE)

def encode_texts(texts: list[str]) -> torch.Tensor:
    model.encoder.eval()
    embeddings = []
    for start in tqdm(range(0, len(texts), ENCODE_BATCH_SIZE), desc='precompute XLM-R embeddings'):
        batch_texts = texts[start:start + ENCODE_BATCH_SIZE]
        encoded = model.tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.no_grad():
            hidden_states = model.encoder(**encoded).last_hidden_state
            pooled = mean_pool(hidden_states, encoded['attention_mask'])
        embeddings.append(pooled.detach().cpu())
    return torch.cat(embeddings, dim=0)

all_texts = [row['text'] for row in rows]
all_embeddings = encode_texts(all_texts)
all_labels = torch.tensor(labels, dtype=torch.float).unsqueeze(-1)

train_dataset = TensorDataset(all_embeddings[train_indices], all_labels[train_indices])
val_dataset = TensorDataset(all_embeddings[val_indices], all_labels[val_indices])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def compute_metrics(probabilities: list[float], labels: list[float]) -> dict[str, float | None]:
    predictions = [1.0 if probability >= 0.5 else 0.0 for probability in probabilities]
    correct = sum(prediction == label for prediction, label in zip(predictions, labels))
    accuracy = correct / max(1, len(labels))

    tp = sum(p == 1.0 and y == 1.0 for p, y in zip(predictions, labels))
    fp = sum(p == 1.0 and y == 0.0 for p, y in zip(predictions, labels))
    fn = sum(p == 0.0 and y == 1.0 for p, y in zip(predictions, labels))

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    roc_auc = None
    if len(set(labels)) > 1:
        roc_auc = float(roc_auc_score(labels, probabilities))

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': roc_auc}


def evaluate_classifier() -> dict[str, float | None]:
    model.classifier.eval()
    losses = []
    probabilities = []
    gold = []
    with torch.no_grad():
        for embeddings, batch_labels in tqdm(val_loader, desc='val', leave=False):
            embeddings = embeddings.to(device)
            batch_labels = batch_labels.to(device)
            logits = model.classifier(embeddings)
            loss = criterion(logits, batch_labels)
            losses.append(float(loss.item()) * batch_labels.size(0))
            probabilities.extend(torch.sigmoid(logits).squeeze(-1).detach().cpu().tolist())
            gold.extend(batch_labels.squeeze(-1).detach().cpu().tolist())
    metrics = compute_metrics(probabilities, gold)
    metrics['loss'] = sum(losses) / max(1, len(gold))
    return metrics

In [ ]:
best_f1 = -1.0
best_epoch = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.classifier.train()
    total_loss = 0.0
    total_examples = 0
    progress = tqdm(train_loader, desc=f'train epoch {epoch}/{EPOCHS}', leave=False)
    for embeddings, batch_labels in progress:
        embeddings = embeddings.to(device)
        batch_labels = batch_labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model.classifier(embeddings)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * batch_labels.size(0)
        total_examples += batch_labels.size(0)
        progress.set_postfix(loss=f'{loss.item():.4f}')

    train_loss = total_loss / max(1, total_examples)
    val_metrics = evaluate_classifier()
    val_f1 = float(val_metrics['f1'] or 0.0)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val': val_metrics})

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        torch.save({
            'epoch': epoch,
            'classifier_state_dict': model.classifier.state_dict(),
            'metrics': val_metrics,
        }, OUTPUT_DIR / 'idiom_signal.pt')

    roc_auc = val_metrics['roc_auc']
    roc_auc_text = 'NA' if roc_auc is None else f'{roc_auc:.4f}'
    print(
        f'epoch={epoch}/{EPOCHS} '
        f'train_loss={train_loss:.4f} '
        f"val_loss={float(val_metrics['loss']):.4f} "
        f"val_acc={float(val_metrics['accuracy']):.4f} "
        f"val_f1={float(val_metrics['f1']):.4f} "
        f'val_roc_auc={roc_auc_text}'
    )

print('best_epoch:', best_epoch)
print('best_val_f1:', best_f1)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.tokenizer.save_pretrained(OUTPUT_DIR / 'tokenizer')

config = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'dropout': DROPOUT,
    'encoder_frozen_during_training': True,
    'train_size': len(train_indices),
    'val_size': len(val_indices),
    'seed': SEED,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LEARNING_RATE,
    'best_epoch': best_epoch,
    'best_val_f1': best_f1,
}
with (OUTPUT_DIR / 'config.json').open('w', encoding='utf-8') as file:
    json.dump(config, file, indent=2, ensure_ascii=False)
with (OUTPUT_DIR / 'metrics.json').open('w', encoding='utf-8') as file:
    json.dump({'best_epoch': best_epoch, 'best_val_f1': best_f1, 'history': history}, file, indent=2, ensure_ascii=False)

print('saved:', OUTPUT_DIR)

In [ ]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH).replace('.zip', ''), 'zip', OUTPUT_DIR)
print('zip:', ZIP_PATH)

files.download(str(ZIP_PATH))